# TPC-H Lakehouse — Regional Manager

One notebook, top to bottom: **bronze → silver → gold → validation → business questions → monitoring**.

Source: `samples.tpch` (8 tables). Run all cells on any Databricks compute; re-running is safe (every table is overwritten).

**Definitions used everywhere**

| term | definition |
|---|---|
| Net revenue | `extended_price × (1 − discount)`, tax excluded |
| Region of a sale | the region of the **customer** who placed the order |
| Cross-region sale | customer region ≠ supplier region |
| Period | order date; month-over-month comparisons use **complete** months only |

## 0 · Setup
Nothing below hard-codes a catalog or schema. To move to another workspace or environment, change the widgets.

In [ ]:
from pyspark.sql import functions as F, Window
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("env", "dev", "Environment (dev / preprod / prod)")
dbutils.widgets.text("source", "samples.tpch", "Source schema")

CATALOG = dbutils.widgets.get("catalog")
ENV     = dbutils.widgets.get("env")
SOURCE  = dbutils.widgets.get("source")
FORMAT  = "delta"

# Naming rule: <catalog>.tpch_<env>_<layer>.<table>
prefix = f"{CATALOG}." if CATALOG else ""
BRONZE, SILVER, GOLD = (f"{prefix}tpch_{ENV}_{layer}" for layer in ("bronze", "silver", "gold"))
for schema in (BRONZE, SILVER, GOLD):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

TABLES = ["region", "nation", "customer", "supplier", "part", "partsupp", "orders", "lineitem"]  # parents first


def save(df, table, mode="overwrite"):
    df.write.format(FORMAT).mode(mode).option("overwriteSchema", "true").saveAsTable(table)


def sql(query):
    """spark.sql with {bronze} / {silver} / {gold} / {source} placeholders."""
    return spark.sql(query.format(bronze=BRONZE, silver=SILVER, gold=GOLD, source=SOURCE))


print(BRONZE, SILVER, GOLD, sep="\n")

## 1 · Bronze — the source, as is
No renames, casts or filters. Only two metadata columns are added for lineage.

In [ ]:
for t in TABLES:
    save(
        spark.table(f"{SOURCE}.{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_table", F.lit(f"{SOURCE}.{t}")),
        f"{BRONZE}.{t}",
    )
display(sql("SHOW TABLES IN {bronze}"))

## 2 · Silver — 3NF with enforced data quality

TPC-H is already normalised, so silver keeps its eight entities: every non-key column depends on the key, the whole key
(`partsupp`, `lineitem` have composite keys) and nothing but the key. Nation and region names are not repeated on
customers or suppliers; they are reached through `nation_key → region_key`. Silver changes names and types only
(`c_custkey` → `customer_key`, money as `decimal(18,2)`, strings trimmed).

Each table is declared once below: columns, primary key, NOT NULL columns, CHECK rules, foreign keys.
Rows that break a rule go to `<table>_quarantine` with the names of the rules they broke, so nothing is dropped silently.

In [ ]:
MONEY = "decimal(18,2)"
# source column -> (silver column, type);  fks: (columns, parent table) - parent columns have the same names
SPECS = {
    "region": dict(
        cols={"r_regionkey": ("region_key", "bigint"), "r_name": ("name", "string"), "r_comment": ("comment", "string")},
        pk=["region_key"], not_null=["name"], checks={}, fks=[]),
    "nation": dict(
        cols={"n_nationkey": ("nation_key", "bigint"), "n_name": ("name", "string"),
              "n_regionkey": ("region_key", "bigint"), "n_comment": ("comment", "string")},
        pk=["nation_key"], not_null=["name", "region_key"], checks={}, fks=[(["region_key"], "region")]),
    "customer": dict(
        cols={"c_custkey": ("customer_key", "bigint"), "c_name": ("name", "string"), "c_address": ("address", "string"),
              "c_nationkey": ("nation_key", "bigint"), "c_phone": ("phone", "string"),
              "c_acctbal": ("account_balance", MONEY), "c_mktsegment": ("market_segment", "string"),
              "c_comment": ("comment", "string")},
        pk=["customer_key"], not_null=["nation_key", "account_balance"], checks={}, fks=[(["nation_key"], "nation")]),
    "supplier": dict(
        cols={"s_suppkey": ("supplier_key", "bigint"), "s_name": ("name", "string"), "s_address": ("address", "string"),
              "s_nationkey": ("nation_key", "bigint"), "s_phone": ("phone", "string"),
              "s_acctbal": ("account_balance", MONEY), "s_comment": ("comment", "string")},
        pk=["supplier_key"], not_null=["nation_key"], checks={}, fks=[(["nation_key"], "nation")]),
    "part": dict(
        cols={"p_partkey": ("part_key", "bigint"), "p_name": ("name", "string"), "p_mfgr": ("manufacturer", "string"),
              "p_brand": ("brand", "string"), "p_type": ("type", "string"), "p_size": ("size", "int"),
              "p_container": ("container", "string"), "p_retailprice": ("retail_price", MONEY),
              "p_comment": ("comment", "string")},
        pk=["part_key"], not_null=["name"], checks={"retail_price_positive": "retail_price > 0"}, fks=[]),
    "partsupp": dict(
        cols={"ps_partkey": ("part_key", "bigint"), "ps_suppkey": ("supplier_key", "bigint"),
              "ps_availqty": ("available_quantity", "int"), "ps_supplycost": ("supply_cost", MONEY),
              "ps_comment": ("comment", "string")},
        pk=["part_key", "supplier_key"], not_null=[], checks={"supply_cost_not_negative": "supply_cost >= 0"},
        fks=[(["part_key"], "part"), (["supplier_key"], "supplier")]),
    "orders": dict(
        cols={"o_orderkey": ("order_key", "bigint"), "o_custkey": ("customer_key", "bigint"),
              "o_orderstatus": ("order_status", "string"), "o_totalprice": ("total_price", MONEY),
              "o_orderdate": ("order_date", "date"), "o_orderpriority": ("order_priority", "string"),
              "o_clerk": ("clerk", "string"), "o_shippriority": ("ship_priority", "int"),
              "o_comment": ("comment", "string")},
        pk=["order_key"], not_null=["customer_key", "order_date"], checks={}, fks=[(["customer_key"], "customer")]),
    "lineitem": dict(
        cols={"l_orderkey": ("order_key", "bigint"), "l_linenumber": ("line_number", "int"),
              "l_partkey": ("part_key", "bigint"), "l_suppkey": ("supplier_key", "bigint"),
              "l_quantity": ("quantity", MONEY), "l_extendedprice": ("extended_price", MONEY),
              "l_discount": ("discount", MONEY), "l_tax": ("tax", MONEY),
              "l_returnflag": ("return_flag", "string"), "l_linestatus": ("line_status", "string"),
              "l_shipdate": ("ship_date", "date"), "l_commitdate": ("commit_date", "date"),
              "l_receiptdate": ("receipt_date", "date"), "l_shipinstruct": ("ship_instructions", "string"),
              "l_shipmode": ("ship_mode", "string"), "l_comment": ("comment", "string")},
        pk=["order_key", "line_number"], not_null=["quantity", "extended_price", "discount"],
        checks={"quantity_positive": "quantity > 0", "extended_price_positive": "extended_price > 0",
                "discount_between_0_and_1": "discount >= 0 AND discount <= 1"},
        # (part_key, supplier_key) is a two-column foreign key into partsupp
        fks=[(["order_key"], "orders"), (["part_key", "supplier_key"], "partsupp")]),
}

In [ ]:
def evaluate_rules(table, bronze_df=None):
    """Typed silver columns + `_failed_rules` (empty array = clean row)."""
    spec = SPECS[table]
    df = bronze_df if bronze_df is not None else spark.table(f"{BRONZE}.{table}")
    df = df.select(
        *[(F.trim(F.col(src)) if dtype == "string" else F.col(src).cast(dtype)).alias(name)
          for src, (name, dtype) in spec["cols"].items()],
        "_ingested_at",
    )
    rules = []  # (rule name, condition that must be TRUE)

    # primary key: not null and unique (duplicates are quarantined, one copy is kept)
    rules += [(f"not_null_{c}", F.col(c).isNotNull()) for c in spec["pk"] + spec["not_null"]]
    df = df.withColumn("_rn", F.row_number().over(Window.partitionBy(*spec["pk"]).orderBy(F.col("_ingested_at").desc())))
    rules.append(("pk_unique", F.col("_rn") == 1))

    rules += [(name, F.coalesce(F.expr(expr), F.lit(False))) for name, expr in spec["checks"].items()]

    # foreign keys against the already-cleaned silver parent (parent key is unique, so no fan-out)
    for i, (cols, parent) in enumerate(spec["fks"]):
        p = spark.table(f"{SILVER}.{parent}").select(*[F.col(c).alias(f"_p{i}_{c}") for c in cols], F.lit(True).alias(f"_hit{i}"))
        df = df.join(p, [df[c] == p[f"_p{i}_{c}"] for c in cols], "left")
        rules.append((f"fk_{'_'.join(cols)}__{parent}", F.col(f"_hit{i}").isNotNull()))

    failed = F.filter(F.array(*[F.when(~cond, F.lit(name)) for name, cond in rules]), lambda x: x.isNotNull())
    return df.select(*[name for name, _ in spec["cols"].values()], failed.alias("_failed_rules"))


for t in TABLES:   # parents before children
    rows = evaluate_rules(t)
    save(rows.filter(F.size("_failed_rules") == 0).drop("_failed_rules"), f"{SILVER}.{t}")
    save(rows.filter(F.size("_failed_rules") > 0), f"{SILVER}.{t}_quarantine")

display(spark.createDataFrame(
    [(t, spark.table(f"{BRONZE}.{t}").count(), spark.table(f"{SILVER}.{t}").count(),
      spark.table(f"{SILVER}.{t}_quarantine").count()) for t in TABLES],
    "table string, bronze_rows long, silver_rows long, quarantined_rows long"))

### Constraints on the tables
`NOT NULL` and `CHECK` are enforced by Delta on every later write. `PRIMARY KEY` / `FOREIGN KEY` are informational in
Unity Catalog (not enforced by the engine), which is why the rules above exist. They do draw the **ER diagram**:
Catalog Explorer → open `lineitem` in the silver schema → **View relationships** (screenshot for the presentation).

In [ ]:
statements = []
for t in reversed(TABLES):
    statements += [f"ALTER TABLE {SILVER}.{t} DROP CONSTRAINT IF EXISTS fk_{'_'.join(cols)}__{parent}" for cols, parent in SPECS[t]["fks"]]
for t in TABLES:
    spec = SPECS[t]
    statements.append(f"ALTER TABLE {SILVER}.{t} DROP CONSTRAINT IF EXISTS pk_{t}")
    statements += [f"ALTER TABLE {SILVER}.{t} ALTER COLUMN {c} SET NOT NULL" for c in spec["pk"] + spec["not_null"]]
    for name, expr in spec["checks"].items():
        statements.append(f"ALTER TABLE {SILVER}.{t} DROP CONSTRAINT IF EXISTS chk_{name}")
        statements.append(f"ALTER TABLE {SILVER}.{t} ADD CONSTRAINT chk_{name} CHECK ({expr})")
    statements.append(f"ALTER TABLE {SILVER}.{t} ADD CONSTRAINT pk_{t} PRIMARY KEY ({', '.join(spec['pk'])})")
for t in TABLES:
    statements += [f"ALTER TABLE {SILVER}.{t} ADD CONSTRAINT fk_{'_'.join(cols)}__{parent} "
                   f"FOREIGN KEY ({', '.join(cols)}) REFERENCES {SILVER}.{parent} ({', '.join(cols)})"
                   for cols, parent in SPECS[t]["fks"]]

log = []
for stmt in statements:
    try:
        spark.sql(stmt)
        log.append((stmt, "OK"))
    except Exception as e:   # e.g. PK/FK need Unity Catalog - report, don't stop
        log.append((stmt, "SKIPPED: " + str(e).splitlines()[0][:150]))
display(spark.createDataFrame(log, "statement string, result string"))

## 3 · Gold — a star schema for regional questions

| table | grain | used for |
|---|---|---|
| `dim_geography` | nation | nation → region hierarchy |
| `dim_customer`, `dim_supplier`, `dim_part` | one row per entity | Q2, labels |
| `fact_sales` | order line | Q3 cross-region share; base of the aggregates |
| `agg_region_revenue_monthly` | region × month | Q3 ranking, monitoring |
| `agg_part_region_revenue` | part × region | Q1 |
| `agg_part_coverage` | part | Q4 |

`fact_sales` carries both the customer's and the supplier's nation and region, so the two roles are never mixed up.
The inner joins are safe because silver guarantees every key resolves; validation proves no row was lost or duplicated.

In [ ]:
GOLD_TABLES = {
    "dim_geography": """
        SELECT n.nation_key, n.name AS nation_name, r.region_key, r.name AS region_name
        FROM {silver}.nation n JOIN {silver}.region r ON n.region_key = r.region_key""",
    "dim_customer": """
        SELECT c.customer_key, c.name AS customer_name, c.market_segment, c.account_balance,
               g.nation_key, g.nation_name, g.region_key, g.region_name
        FROM {silver}.customer c JOIN {gold}.dim_geography g ON c.nation_key = g.nation_key""",
    "dim_supplier": """
        SELECT s.supplier_key, s.name AS supplier_name, g.nation_key, g.nation_name, g.region_key, g.region_name
        FROM {silver}.supplier s JOIN {gold}.dim_geography g ON s.nation_key = g.nation_key""",
    "dim_part": """
        SELECT part_key, name AS part_name, manufacturer, brand, type, size, container, retail_price
        FROM {silver}.part""",
    "fact_sales": """
        SELECT l.order_key, l.line_number, o.order_date,
               CAST(date_trunc('MONTH', o.order_date) AS DATE) AS order_month,
               o.customer_key, l.part_key, l.supplier_key,
               c.nation_name AS customer_nation, c.region_key AS customer_region_key, c.region_name AS customer_region,
               s.nation_name AS supplier_nation, s.region_key AS supplier_region_key, s.region_name AS supplier_region,
               c.region_key <> s.region_key AS is_cross_region,
               l.quantity, l.extended_price, l.discount,
               CAST(l.extended_price * (1 - l.discount) AS DECIMAL(18,4)) AS net_revenue
        FROM {silver}.lineitem l
        JOIN {silver}.orders o     ON l.order_key    = o.order_key
        JOIN {gold}.dim_customer c ON o.customer_key = c.customer_key
        JOIN {gold}.dim_supplier s ON l.supplier_key = s.supplier_key""",
    "agg_region_revenue_monthly": """
        WITH bounds AS (SELECT min(order_date) AS min_date, max(order_date) AS max_date FROM {gold}.fact_sales),
        monthly AS (
            SELECT customer_region_key AS region_key, customer_region AS region_name, order_month,
                   SUM(net_revenue) AS revenue, COUNT(DISTINCT order_key) AS orders, COUNT(*) AS line_items
            FROM {gold}.fact_sales GROUP BY customer_region_key, customer_region, order_month)
        SELECT m.*, day(last_day(m.order_month)) AS days_in_month,
               -- a month only counts once the data covers all of it
               (m.order_month >= b.min_date AND last_day(m.order_month) <= b.max_date) AS is_complete_month,
               RANK() OVER (PARTITION BY m.order_month ORDER BY m.revenue DESC) AS revenue_rank,
               CAST(m.revenue / SUM(m.revenue) OVER (PARTITION BY m.order_month) AS DOUBLE) AS revenue_share
        FROM monthly m CROSS JOIN bounds b""",
    "agg_part_region_revenue": """
        SELECT part_key, customer_region AS region_name, SUM(net_revenue) AS revenue, COUNT(*) AS line_items
        FROM {gold}.fact_sales GROUP BY part_key, customer_region""",
    "agg_part_coverage": """
        WITH sold AS (
            SELECT part_key, COUNT(*) AS regions_sold_in,
                   concat_ws(', ', sort_array(collect_list(region_name))) AS sold_in_regions,
                   SUM(line_items) AS line_items, SUM(revenue) AS revenue
            FROM {gold}.agg_part_region_revenue GROUP BY part_key),
        supply AS (
            SELECT ps.part_key, COUNT(DISTINCT s.region_key) AS supplier_regions
            FROM {silver}.partsupp ps JOIN {gold}.dim_supplier s ON ps.supplier_key = s.supplier_key
            GROUP BY ps.part_key)
        SELECT p.part_key, p.part_name, p.brand,
               COALESCE(sold.regions_sold_in, 0) AS regions_sold_in, COALESCE(sold.sold_in_regions, '') AS sold_in_regions,
               COALESCE(sold.line_items, 0) AS line_items, COALESCE(sold.revenue, 0) AS revenue,
               COALESCE(supply.supplier_regions, 0) AS supplier_regions
        FROM {gold}.dim_part p
        LEFT JOIN sold   ON p.part_key = sold.part_key
        LEFT JOIN supply ON p.part_key = supply.part_key""",
}
for name, query in GOLD_TABLES.items():   # in order: later tables read earlier ones
    save(sql(query), f"{GOLD}.{name}")
display(sql("SHOW TABLES IN {gold}"))

## 4 · Validation — can the regional numbers be trusted?

Every check is one SQL statement that returns the **number of violations** (0 = pass). Results are appended to
`gold.dq_results`; the last cell of this section fails the run if anything is violated.

| group | proves |
|---|---|
| GEOGRAPHY | every customer and supplier → valid nation → valid region (profile rule 1) |
| REVENUE | regional revenue sums to the total, identical in all three layers, no double counting (profile rule 2) |
| COMPLETENESS | no row lost or duplicated source → bronze → silver → gold |

In [ ]:
B_REV = "SUM(CAST(l_extendedprice * (1 - l_discount) AS DECIMAL(18,4)))"   # net revenue on raw columns
S_REV = "SUM(CAST(extended_price * (1 - discount) AS DECIMAL(18,4)))"      # same definition on silver
TOL = "0.01"
checks = []   # (group, name, SQL returning number of violations)

for t in TABLES:
    pk = ", ".join(SPECS[t]["pk"])
    checks += [
        ("COMPLETENESS", f"bronze_matches_source__{t}",
         f"SELECT abs((SELECT count(*) FROM {{source}}.{t}) - (SELECT count(*) FROM {{bronze}}.{t}))"),
        ("COMPLETENESS", f"silver_plus_quarantine_equals_bronze__{t}",
         f"SELECT abs((SELECT count(*) FROM {{bronze}}.{t}) - (SELECT count(*) FROM {{silver}}.{t}) - (SELECT count(*) FROM {{silver}}.{t}_quarantine))"),
        ("COMPLETENESS", f"silver_pk_unique__{t}",
         f"SELECT count(*) FROM (SELECT {pk} FROM {{silver}}.{t} GROUP BY {pk} HAVING count(*) > 1)"),
    ]
checks += [
    ("COMPLETENESS", "fact_rowcount_equals_lineitem",
     "SELECT abs((SELECT count(*) FROM {silver}.lineitem) - (SELECT count(*) FROM {gold}.fact_sales))"),
    ("COMPLETENESS", "fact_grain_unique",
     "SELECT count(*) FROM (SELECT 1 FROM {gold}.fact_sales GROUP BY order_key, line_number HAVING count(*) > 1)"),
    ("COMPLETENESS", "dim_customer_keeps_customers_without_orders",
     "SELECT abs((SELECT count(*) FROM {silver}.customer) - (SELECT count(*) FROM {gold}.dim_customer))"),

    # GEOGRAPHY: on the raw data (a statement about the source) and on silver (must hold by construction)
    ("GEOGRAPHY", "bronze_customer_has_valid_nation",
     "SELECT count(*) FROM {bronze}.customer c LEFT ANTI JOIN {bronze}.nation n ON c.c_nationkey = n.n_nationkey"),
    ("GEOGRAPHY", "bronze_supplier_has_valid_nation",
     "SELECT count(*) FROM {bronze}.supplier s LEFT ANTI JOIN {bronze}.nation n ON s.s_nationkey = n.n_nationkey"),
    ("GEOGRAPHY", "bronze_nation_has_valid_region",
     "SELECT count(*) FROM {bronze}.nation n LEFT ANTI JOIN {bronze}.region r ON n.n_regionkey = r.r_regionkey"),
    ("GEOGRAPHY", "silver_customer_has_valid_nation",
     "SELECT count(*) FROM {silver}.customer c LEFT ANTI JOIN {silver}.nation n ON c.nation_key = n.nation_key"),
    ("GEOGRAPHY", "silver_supplier_has_valid_nation",
     "SELECT count(*) FROM {silver}.supplier s LEFT ANTI JOIN {silver}.nation n ON s.nation_key = n.nation_key"),
    ("GEOGRAPHY", "silver_nation_has_valid_region",
     "SELECT count(*) FROM {silver}.nation n LEFT ANTI JOIN {silver}.region r ON n.region_key = r.region_key"),
    ("GEOGRAPHY", "geography_quarantine_is_empty",
     "SELECT (SELECT count(*) FROM {silver}.region_quarantine) + (SELECT count(*) FROM {silver}.nation_quarantine)"
     " + (SELECT count(*) FROM {silver}.customer_quarantine) + (SELECT count(*) FROM {silver}.supplier_quarantine)"),
    ("GEOGRAPHY", "nation_in_exactly_one_region",
     "SELECT count(*) FROM (SELECT nation_key FROM {gold}.dim_geography GROUP BY nation_key HAVING count(*) > 1)"),
    ("GEOGRAPHY", "fact_has_both_regions",
     "SELECT count(*) FROM {gold}.fact_sales WHERE customer_region_key IS NULL OR supplier_region_key IS NULL"),

    # REVENUE: exact decimal arithmetic, so layers must agree to the cent
    ("REVENUE", "net_revenue_bronze_equals_silver",
     f"SELECT CASE WHEN abs((SELECT {B_REV} FROM {{bronze}}.lineitem) - (SELECT {S_REV} FROM {{silver}}.lineitem)) > {TOL} THEN 1 ELSE 0 END"),
    ("REVENUE", "net_revenue_silver_equals_gold",
     f"SELECT CASE WHEN abs((SELECT {S_REV} FROM {{silver}}.lineitem) - (SELECT SUM(net_revenue) FROM {{gold}}.fact_sales)) > {TOL} THEN 1 ELSE 0 END"),
    ("REVENUE", "regions_sum_to_total",
     f"SELECT CASE WHEN abs((SELECT SUM(revenue) FROM {{gold}}.agg_region_revenue_monthly) - (SELECT SUM(net_revenue) FROM {{gold}}.fact_sales)) > {TOL} THEN 1 ELSE 0 END"),
    ("REVENUE", "part_region_sums_to_total",
     f"SELECT CASE WHEN abs((SELECT SUM(revenue) FROM {{gold}}.agg_part_region_revenue) - (SELECT SUM(net_revenue) FROM {{gold}}.fact_sales)) > {TOL} THEN 1 ELSE 0 END"),
    ("REVENUE", "each_sale_counted_in_one_region",
     "SELECT abs((SELECT SUM(line_items) FROM {gold}.agg_region_revenue_monthly) - (SELECT count(*) FROM {gold}.fact_sales))"),
    ("REVENUE", "region_revenue_matches_recompute_from_bronze",
     f"""SELECT count(*) FROM (
            SELECT trim(r.r_name) AS region_name, {B_REV} AS revenue
            FROM {{bronze}}.lineitem l
            JOIN {{bronze}}.orders o   ON l.l_orderkey  = o.o_orderkey
            JOIN {{bronze}}.customer c ON o.o_custkey   = c.c_custkey
            JOIN {{bronze}}.nation n   ON c.c_nationkey = n.n_nationkey
            JOIN {{bronze}}.region r   ON n.n_regionkey = r.r_regionkey
            GROUP BY trim(r.r_name)) b
         FULL OUTER JOIN (SELECT region_name, SUM(revenue) AS revenue FROM {{gold}}.agg_region_revenue_monthly GROUP BY region_name) g
           ON b.region_name = g.region_name
         WHERE b.revenue IS NULL OR g.revenue IS NULL OR abs(b.revenue - g.revenue) > {TOL}"""),
    ("REVENUE", "monthly_shares_sum_to_one",
     "SELECT count(*) FROM (SELECT order_month FROM {gold}.agg_region_revenue_monthly GROUP BY order_month HAVING abs(SUM(revenue_share) - 1) > 1e-9)"),
]

results = [(g, name, int(sql(q).first()[0] or 0)) for g, name, q in checks]
dq = (spark.createDataFrame(results, "check_group string, check_name string, violations long")
      .withColumn("status", F.when(F.col("violations") == 0, "PASS").otherwise("FAIL"))
      .withColumn("run_at", F.current_timestamp()))
save(dq, f"{GOLD}.dq_results", mode="append")
print(f"{sum(v == 0 for _, _, v in results)}/{len(results)} checks passed")
display(dq.orderBy("check_group", "check_name"))

### The same net revenue in every layer

In [ ]:
display(sql(f"""
    SELECT 'bronze' AS layer, {B_REV} AS net_revenue FROM {{bronze}}.lineitem
    UNION ALL SELECT 'silver', {S_REV} FROM {{silver}}.lineitem
    UNION ALL SELECT 'gold: fact_sales', SUM(net_revenue) FROM {{gold}}.fact_sales
    UNION ALL SELECT 'gold: sum of regions', SUM(revenue) FROM {{gold}}.agg_region_revenue_monthly
"""))

### Do the rules actually catch bad data?
The source is clean, so passing checks prove little. We inject three broken customers into a copy of bronze
(unknown nation, duplicate key, missing nation) and run the silver rules on it. Nothing is written.

In [ ]:
bronze_customer = spark.table(f"{BRONZE}.customer")
sample = [r.asDict() for r in bronze_customer.limit(3).collect()]
bad = spark.createDataFrame(
    [sample[0] | {"c_custkey": -1, "c_nationkey": 999},     # nation does not exist
     sample[1],                                               # duplicate primary key
     sample[2] | {"c_custkey": -2, "c_nationkey": None}],    # nation missing
    schema=bronze_customer.schema)
display(evaluate_rules("customer", bronze_customer.unionByName(bad))
        .filter(F.size("_failed_rules") > 0)
        .select("customer_key", "name", "nation_key", "_failed_rules"))

In [ ]:
failed = [name for _, name, v in results if v > 0]
assert not failed, f"Data quality checks failed: {failed}"
print("All data quality checks passed.")

## 5 · Business questions
Every answer reads **gold only**.

In [ ]:
COLORS = {"AFRICA": "#E69F00", "AMERICA": "#0072B2", "ASIA": "#D55E00", "EUROPE": "#009E73", "MIDDLE EAST": "#CC79A7"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

### Q1 · What are the top 5 products by revenue in Asia?

In [ ]:
q1 = sql("""
    SELECT p.part_key, p.part_name, p.brand, CAST(a.revenue AS DOUBLE) AS revenue, a.line_items
    FROM {gold}.agg_part_region_revenue a JOIN {gold}.dim_part p ON a.part_key = p.part_key
    WHERE a.region_name = 'ASIA'
    ORDER BY a.revenue DESC, p.part_key
    LIMIT 5""")
display(q1)

pdf = q1.toPandas()
fig, ax = plt.subplots(figsize=(9, 3.4))
bars = ax.barh([f"#{k} · {n}" for k, n in zip(pdf.part_key, pdf.part_name)][::-1], pdf.revenue[::-1], color=COLORS["ASIA"])
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in pdf.revenue[::-1]], padding=3)
ax.set_xlim(0, pdf.revenue.max() * 1.15); ax.set_xlabel("Net revenue")
ax.set_title("Top 5 products by net revenue — ASIA", loc="left")
plt.tight_layout(); plt.show()

### Q2 · How many customers are in India, and what is their average account balance?

In [ ]:
display(sql("""
    SELECT COUNT(*) AS customers, CAST(AVG(account_balance) AS DECIMAL(18,2)) AS avg_account_balance
    FROM {gold}.dim_customer
    WHERE nation_name = 'INDIA'"""))

### Q3 · Which region has the highest total revenue, and what share of sales is cross-region?

In [ ]:
q3a = sql("""
    SELECT region_name, CAST(SUM(revenue) AS DECIMAL(22,2)) AS revenue,
           CAST(SUM(revenue) / SUM(SUM(revenue)) OVER () AS DOUBLE) AS revenue_share,
           RANK() OVER (ORDER BY SUM(revenue) DESC) AS revenue_rank
    FROM {gold}.agg_region_revenue_monthly
    GROUP BY region_name
    ORDER BY revenue DESC""")
display(q3a)

display(sql("""
    SELECT CAST(SUM(CASE WHEN is_cross_region THEN net_revenue ELSE 0 END) AS DECIMAL(22,2)) AS cross_region_revenue,
           CAST(SUM(net_revenue) AS DECIMAL(22,2)) AS total_revenue,
           CAST(SUM(CASE WHEN is_cross_region THEN net_revenue ELSE 0 END) / SUM(net_revenue) AS DOUBLE) AS cross_region_share
    FROM {gold}.fact_sales"""))

q3b = sql("""
    SELECT customer_region AS region_name,
           CAST(SUM(CASE WHEN is_cross_region THEN net_revenue ELSE 0 END) / SUM(net_revenue) AS DOUBLE) AS cross_region_share
    FROM {gold}.fact_sales GROUP BY customer_region ORDER BY customer_region""")

In [ ]:
a = q3a.selectExpr("region_name", "CAST(revenue AS DOUBLE) AS revenue").toPandas()
b = q3b.toPandas()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
bars = axes[0].bar(a.region_name, a.revenue, color=[COLORS[r] for r in a.region_name])
axes[0].bar_label(bars, labels=[f"{v / 1e9:.2f}B" for v in a.revenue], fontsize=9)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1e9:.0f}B"))
axes[0].set_title("Total net revenue by region", loc="left")
bars = axes[1].bar(b.region_name, b.cross_region_share, color=[COLORS[r] for r in b.region_name])
axes[1].bar_label(bars, labels=[f"{v:.1%}" for v in b.cross_region_share], fontsize=9)
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[1].set_ylim(0, 1)
axes[1].set_title("Share of revenue supplied from another region", loc="left")
for ax in axes: ax.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

### Q4 · Are there products sold in only one or two regions? What might that indicate?
Two views of coverage: where **customers** buy a product, and where it can be **sourced** from (`partsupp` → supplier region).

In [ ]:
sold = sql("SELECT regions_sold_in, COUNT(*) AS products, AVG(line_items) AS avg_line_items FROM {gold}.agg_part_coverage GROUP BY regions_sold_in ORDER BY regions_sold_in")
display(sold)
display(sql("SELECT * FROM {gold}.agg_part_coverage WHERE regions_sold_in BETWEEN 1 AND 2 ORDER BY regions_sold_in, part_key"))
sourced = sql("SELECT supplier_regions, COUNT(*) AS products FROM {gold}.agg_part_coverage GROUP BY supplier_regions ORDER BY supplier_regions")
display(sourced)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, df, col, title in [(axes[0], sold, "regions_sold_in", "Regions a product is SOLD in"),
                           (axes[1], sourced, "supplier_regions", "Regions a product can be SOURCED from")]:
    counts = {i: 0 for i in range(6)} | {int(r[col]): r["products"] for r in df.collect()}
    bars = ax.bar([str(k) for k in counts], list(counts.values()), color=["#D55E00" if k <= 2 else "#BBBBBB" for k in counts])
    ax.bar_label(bars, fmt="{:,.0f}", fontsize=8); ax.set_title(title, loc="left"); ax.set_xlabel("number of regions")
axes[0].set_ylabel("products")
plt.tight_layout(); plt.show()

## 6 · Monitoring and alerting — revenue by region over time

Monitored table: `gold.agg_region_revenue_monthly`. Two alert rules, on **complete months only**:

* **VOLUME** — revenue *per day* deviates ≥ 5% from the region's trailing 3-month average
  (per day, because February is ~10% shorter than January).
* **RANK** — the region's rank changed *and* its revenue share moved ≥ 1 percentage point from its trailing 3-month average.
  The regions are within ~1% of each other, so rank alone flips almost every month.

In [ ]:
VOLUME_THRESHOLD = 0.05   # |revenue per day vs trailing 3-month average|
SHARE_THRESHOLD  = 0.01   # |revenue share vs trailing 3-month average|, in share points


def evaluate_alerts(monthly):
    """Score every (region, complete month). A pure function, so it can be demonstrated on modified data."""
    by_month  = Window.partitionBy("order_month")
    by_region = Window.partitionBy("region_key").orderBy("order_month")
    trailing3 = by_region.rowsBetween(-3, -1)
    s = (monthly.filter("is_complete_month")
         .withColumn("revenue", F.col("revenue").cast("double"))
         .withColumn("revenue_per_day", F.col("revenue") / F.col("days_in_month"))
         .withColumn("revenue_rank", F.rank().over(by_month.orderBy(F.col("revenue").desc())))
         .withColumn("revenue_share", F.col("revenue") / F.sum("revenue").over(by_month))
         .withColumn("has_baseline", F.count("*").over(trailing3) == 3)
         .withColumn("volume_change", F.col("revenue_per_day") / F.avg("revenue_per_day").over(trailing3) - 1)
         .withColumn("share_change", F.col("revenue_share") - F.avg("revenue_share").over(trailing3))
         .withColumn("rank_change", F.col("revenue_rank") - F.lag("revenue_rank").over(by_region))
         .withColumn("is_latest_month", F.col("order_month") == F.max("order_month").over(Window.partitionBy(F.lit(1)))))
    volume = F.col("has_baseline") & (F.abs("volume_change") >= VOLUME_THRESHOLD)
    rank   = F.col("has_baseline") & (F.col("rank_change") != 0) & (F.abs("share_change") >= SHARE_THRESHOLD)
    alert = F.concat_ws(", ",
        F.when(volume & (F.col("volume_change") < 0), "VOLUME_DROP"), F.when(volume & (F.col("volume_change") > 0), "VOLUME_SPIKE"),
        F.when(rank & (F.col("rank_change") > 0), "RANK_LOSS"),       F.when(rank & (F.col("rank_change") < 0), "RANK_GAIN"))
    return s.select("region_name", "order_month", "is_latest_month", F.round("revenue_per_day", 2).alias("revenue_per_day"),
                    F.round("volume_change", 4).alias("volume_change"), F.round("share_change", 4).alias("share_change"),
                    "revenue_rank", "rank_change", F.when(alert != "", alert).alias("alert"))


monthly = spark.table(f"{GOLD}.agg_region_revenue_monthly")
save(evaluate_alerts(monthly), f"{GOLD}.region_revenue_alerts")
scored = spark.table(f"{GOLD}.region_revenue_alerts")

In [ ]:
pdf = scored.orderBy("order_month").toPandas()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for region, g in pdf.groupby("region_name"):
    axes[0].plot(g.order_month, g.revenue_per_day, label=region, color=COLORS[region], linewidth=1.2)
    axes[1].plot(g.order_month, g.revenue_rank, color=COLORS[region], linewidth=1.1)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1e6:,.1f}M"))
axes[0].set_title("Net revenue per day by region (complete months only)", loc="left")
axes[0].legend(ncol=5, frameon=False, fontsize=8, loc="lower center")
axes[1].invert_yaxis(); axes[1].set_yticks([1, 2, 3, 4, 5]); axes[1].set_title("Revenue rank (1 = largest)", loc="left")
plt.tight_layout(); plt.show()

### How noisy is "normal"? This is where the thresholds come from.

In [ ]:
display(scored.selectExpr(
    "round(stddev(volume_change), 4)   AS stddev_volume_change",
    "round(max(abs(volume_change)), 4) AS max_abs_volume_change",
    "round(max(abs(share_change)), 4)  AS max_abs_share_change",
    "sum(CASE WHEN rank_change <> 0 THEN 1 ELSE 0 END) AS rank_changes",
    "count(rank_change)                AS month_to_month_comparisons",
    "sum(CASE WHEN alert IS NOT NULL THEN 1 ELSE 0 END) AS alerts_in_history"))

### Alerts on the real data — latest complete month

In [ ]:
display(scored.filter("is_latest_month").orderBy("revenue_rank"))

### Would the alert catch a real problem?
The data is healthy, so we cut ASIA's revenue in the latest complete month by 30% **in memory** and run the same rule.

In [ ]:
latest = monthly.filter("is_complete_month").agg(F.max("order_month")).first()[0]
shocked = monthly.withColumn("revenue", F.when((F.col("region_name") == "ASIA") & (F.col("order_month") == F.lit(latest)),
                                                F.col("revenue") * F.lit(0.7)).otherwise(F.col("revenue")))
display(evaluate_alerts(shocked).filter("is_latest_month").orderBy("revenue_rank"))

### Databricks SQL alert
Create an alert on the query printed below with condition `alerts > 0`, scheduled after this notebook.

In [ ]:
print(f"""SELECT count(*) AS alerts, concat_ws('; ', collect_list(concat(region_name, ': ', alert))) AS details
FROM {GOLD}.region_revenue_alerts
WHERE is_latest_month AND alert IS NOT NULL""")